In [0]:
import json, time, datetime as dt
from typing import Optional, Dict, Any
import requests

# ---------------- Config ----------------
CATALOG = "canada_business"
SCHEMA  = "bronze"
BRONZE_SUBDIR = "wits_tariff_raw"   # new bronze folder

REPORTER = "CAN"
PARTNER  = "WLD"
PRODUCT  = "ALL"
INDICATOR = "AHS-WGHTD-AVRG"  # weighted average applied tariff rate (WITS)

BASE_URL = (
  "https://wits.worldbank.org/API/V1/SDMX/V21/datasource/tradestats-tariff"
  f"/reporter/{REPORTER}/year/ALL/partner/{PARTNER}/product/{PRODUCT}/indicator/{INDICATOR}"
)

def run_ts_utc() -> str:
    return dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def get_json_with_retry(url: str, retries: int = 5, timeout: int = 30) -> Dict[str, Any]:
    backoff = 2
    last_err = None
    headers = {"User-Agent": "databricks-lakehouse-demo/1.0"}
    for i in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if i == retries - 1:
                break
            time.sleep(backoff)
            backoff *= 2
    raise RuntimeError(f"Request failed after {retries} attempts: {last_err}")

def resolve_schema_root_location(catalog: str, schema: str) -> str:
    df = spark.sql(f"DESCRIBE SCHEMA EXTENDED {catalog}.{schema}").toPandas()
    rows = df.loc[df["database_description_item"] == "RootLocation", "database_description_value"].values
    if len(rows) == 0 or not rows[0]:
        raise RuntimeError(f"RootLocation not found for {catalog}.{schema}")
    return str(rows[0])

def join_uri(base_uri: str, child: str) -> str:
    return base_uri.rstrip("/") + "/" + child.strip("/")

# ---------------- Resolve location ----------------
root_location = resolve_schema_root_location(CATALOG, SCHEMA)
bronze_base = join_uri(root_location, BRONZE_SUBDIR)
data_base = bronze_base.rstrip("/") + "/data"
manifest_base = bronze_base.rstrip("/") + "/manifests"

dbutils.fs.mkdirs(data_base)
dbutils.fs.mkdirs(manifest_base)

ts = run_ts_utc()
url = BASE_URL + "?format=JSON"

payload = get_json_with_retry(url)

out_file = join_uri(data_base, f"wits_tariff_{REPORTER}_{INDICATOR}_{ts}.json")
dbutils.fs.put(out_file, json.dumps(payload, indent=2), overwrite=False)

manifest = {
  "run_ts": ts,
  "catalog": CATALOG,
  "schema": SCHEMA,
  "bronze_base": bronze_base,
  "source": "WITS API (TradeStats-Tariff SDMX)",
  "request": {
    "reporter": REPORTER,
    "partner": PARTNER,
    "product": PRODUCT,
    "indicator": INDICATOR,
    "url": url
  },
  "files": [{"raw_file": out_file}]
}

manifest_path = join_uri(manifest_base, f"run_manifest_{ts}.json")
dbutils.fs.put(manifest_path, json.dumps(manifest, indent=2), overwrite=False)

print("✅ WITS tariff ingest complete")
print("Bronze base:", bronze_base)
print("Raw file:", out_file)
print("Manifest:", manifest_path)